# Milestone 2: Scene Graph Generation (Simplified reproduction of Xu et al. 2017 - IMP)

**Pipeline:**
1. Object detection using pretrained Faster R-CNN (torchvision, COCO weights)
2. Pairwise relationship inference using spatial geometry (simplified message-passing concept)
3. Scene graph construction and visualization (nodes = objects, edges = relationships)
4. Output: subject-predicate-object triplets

Note: Full IMP training requires the complete Visual Genome dataset and days of GPU training. For this milestone we reproduce the *concept* — object nodes + relationship edges — using a pretrained detector plus geometric relationship rules instead of a VG-trained predicate classifier. This limitation is stated explicitly in the report.

In [ ]:
!pip install torchvision networkx matplotlib pillow -q

In [ ]:
import torch
import torchvision
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.transforms import functional as F
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import networkx as nx

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = fasterrcnn_resnet50_fpn(weights='DEFAULT')
model.eval().to(device)

# COCO class names (index 0 is background)
COCO_CLASSES = torchvision.models.detection.FasterRCNN_ResNet50_FPN_Weights.DEFAULT.meta['categories']
print('Model loaded. Classes available:', len(COCO_CLASSES))

In [1]:
from google.colab import files
uploaded = files.upload()
image_path = list(uploaded.keys())[0]
img = Image.open(image_path).convert('RGB')
plt.figure(figsize=(8,6))
plt.imshow(img)
plt.axis('off')
plt.title('Input Image')
plt.show()

ModuleNotFoundError: No module named 'google'

## Step 2: Object Detection (nodes of the scene graph)

In [ ]:
img_tensor = F.to_tensor(img).to(device)
with torch.no_grad():
    prediction = model([img_tensor])[0]

CONF_THRESH = 0.6
boxes, labels, scores = [], [], []
for box, label, score in zip(prediction['boxes'], prediction['labels'], prediction['scores']):
    if score >= CONF_THRESH:
        boxes.append(box.cpu().numpy())
        labels.append(COCO_CLASSES[label.item()])
        scores.append(score.item())

print(f'Detected {len(boxes)} objects above confidence {CONF_THRESH}:')
for l, s in zip(labels, scores):
    print(f'  {l}: {s:.2f}')

# Visualize detections
fig, ax = plt.subplots(1, figsize=(9,7))
ax.imshow(img)
for box, label in zip(boxes, labels):
    x1,y1,x2,y2 = box
    rect = patches.Rectangle((x1,y1), x2-x1, y2-y1, linewidth=2, edgecolor='lime', facecolor='none')
    ax.add_patch(rect)
    ax.text(x1, y1-5, label, color='white', fontsize=10, backgroundcolor='green')
plt.axis('off')
plt.title('Detected Objects (Scene Graph Nodes)')
plt.savefig('detections.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 3: Relationship Inference (edges of the scene graph)
Simplified geometric relationship module: for each object pair, infer a spatial predicate using bounding-box overlap and relative position. This stands in for the learned predicate classifier / message passing step of the original IMP model.

In [ ]:
def iou(b1, b2):
    x1 = max(b1[0], b2[0]); y1 = max(b1[1], b2[1])
    x2 = min(b1[2], b2[2]); y2 = min(b1[3], b2[3])
    inter = max(0, x2-x1) * max(0, y2-y1)
    a1 = (b1[2]-b1[0])*(b1[3]-b1[1])
    a2 = (b2[2]-b2[0])*(b2[3]-b2[1])
    return inter / (a1 + a2 - inter + 1e-6)

def infer_relationship(b1, b2):
    c1 = ((b1[0]+b1[2])/2, (b1[1]+b1[3])/2)
    c2 = ((b2[0]+b2[2])/2, (b2[1]+b2[3])/2)
    overlap = iou(b1, b2)

    if overlap > 0.3:
        if b1[3] <= b2[3]:
            return 'on'
        else:
            return 'under'
    dx, dy = c2[0]-c1[0], c2[1]-c1[1]
    dist = (dx**2 + dy**2) ** 0.5
    diag = ((img.width)**2 + (img.height)**2) ** 0.5
    if dist / diag > 0.5:
        return None  # too far, skip
    if abs(dy) > abs(dx):
        return 'in front of' if dy > 0 else 'behind'
    else:
        return 'right of' if dx > 0 else 'left of'

triplets = []
for i in range(len(boxes)):
    for j in range(len(boxes)):
        if i == j: continue
        rel = infer_relationship(boxes[i], boxes[j])
        if rel:
            triplets.append((labels[i], rel, labels[j]))

# de-duplicate
triplets = list(dict.fromkeys(triplets))
print('Scene graph triplets (subject - predicate - object):')
for t in triplets:
    print(f'  {t[0]} --{t[1]}--> {t[2]}')

## Step 4: Scene Graph Visualization

In [ ]:
G = nx.DiGraph()
for s, p, o in triplets:
    G.add_edge(s, o, label=p)

plt.figure(figsize=(9,7))
pos = nx.spring_layout(G, seed=42, k=1.2)
nx.draw(G, pos, with_labels=True, node_color='lightblue', node_size=2000,
        font_size=9, arrows=True, arrowsize=20)
edge_labels = nx.get_edge_attributes(G, 'label')
nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, font_size=8)
plt.title('Generated Scene Graph')
plt.axis('off')
plt.savefig('scene_graph.png', dpi=150, bbox_inches='tight')
plt.show()